# European Church Tech 2026 — exploratory analysis

5-minute walkthrough of the open dataset. Loads CSVs, runs basic queries, and produces a small chart.

Source: https://gospelchannel.com/european-church-tech-2026  
License: CC-BY-4.0

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../data'

countries = pd.read_csv(f'{DATA_PATH}/country_aggregates.csv')
platforms = pd.read_csv(f'{DATA_PATH}/platforms.csv')

total_churches = countries['total_churches'].sum()
print(f'{len(countries)} countries, {total_churches:,} churches measured, {len(platforms)} platform-country rows')

## 1. Country leaderboard by Facebook adoption

In [ ]:
primary = countries[countries['total_churches'] >= 250].copy()
primary = primary.sort_values('pct_facebook', ascending=False)
primary[['country', 'total_churches', 'pct_facebook', 'pct_youtube', 'pct_cms_detected']]

## 2. Composite digital-adoption score

Equal-weighted average of website, CMS, Facebook, YouTube, and livestream rates.

In [ ]:
primary['composite'] = (
    primary['pct_website']
    + primary['pct_cms_detected']
    + primary['pct_facebook']
    + primary['pct_youtube']
    + primary['pct_livestream']
) / 5

primary = primary.sort_values('composite', ascending=False)
primary[['country', 'composite', 'pct_facebook', 'pct_youtube', 'pct_cms_detected']]

## 3. WordPress dominance per country

In [ ]:
wp = platforms[platforms['platform'] == 'WordPress'].rename(columns={'count': 'wordpress_count'})
totals = platforms.groupby('country')['count'].sum().reset_index().rename(columns={'count': 'total_detected'})
wp_share = wp.merge(totals, on='country')
wp_share['wordpress_share_pct'] = (wp_share['wordpress_count'] / wp_share['total_detected'] * 100).round(1)
wp_share = wp_share.sort_values('wordpress_share_pct', ascending=False)
wp_share[['country', 'wordpress_count', 'total_detected', 'wordpress_share_pct']]

## 4. "Modern DIY" adoption (Squarespace + Wix + Webflow)

In [ ]:
modern = platforms[platforms['platform'].isin(['Squarespace', 'Wix', 'Webflow'])]
modern_share = modern.groupby('country')['count'].sum().reset_index().rename(columns={'count': 'modern_diy_count'})
modern_share = modern_share.merge(totals, on='country')
modern_share['modern_diy_pct'] = (modern_share['modern_diy_count'] / modern_share['total_detected'] * 100).round(1)
modern_share.sort_values('modern_diy_pct', ascending=False)

## 5. A simple chart

In [ ]:
ax = primary.plot(
    kind='barh',
    x='country',
    y='composite',
    figsize=(8, 5),
    color='#a888a0',
    legend=False,
)
ax.set_xlabel('Composite digital adoption (0-100)')
ax.set_ylabel('')
ax.set_title('European Church Tech 2026 — composite score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## What's next

Some ideas using just the aggregates:

- Compare WordPress share to modern-DIY (Wix/Squarespace/Webflow) by country.
- Compute composite scores with different weightings — does Facebook-weighted ranking change leadership?
- Look at livestream concentration (the Netherlands has a notably high rate).
- Cross-reference with public denomination data from country-level church directories.

Found something interesting? Open an issue or write to **hello@gospelchannel.com**.